In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")
import re
import unicodedata
from scipy import stats
import geopandas as gpd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_seq_items", None)

BASE_DIR      = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data"
MENSUALES_DIR = os.path.join(BASE_DIR, "Escenarios Cambio Climatico IDEAM IV comunicacion", "Mensuales")
WEB_DATA_DIR  = os.path.join(BASE_DIR, "Scripts Python", "webpage_climate", "data")

os.chdir(BASE_DIR)
print("Directorio de trabajo :", os.getcwd())
print("Carpeta web/data      :", WEB_DATA_DIR)

Directorio de trabajo : C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data
Carpeta web/data      : C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Scripts Python\webpage_climate\data


In [2]:
pip install geopandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install sodapy

Note: you may need to restart the kernel to use updated packages.


## Precipitación histórica – Carga y procesamiento

In [4]:
precipitacion = pd.read_csv("Precipitación_20251222.csv")

for col in ["Latitud", "Longitud", "ValorObservado"]:
    precipitacion[col] = (
        precipitacion[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .replace("nan", None)
        .astype(float)
    )

fecha_str_ = precipitacion["FechaObservacion"].str.slice(0, 11)
precipitacion["fecha"] = pd.to_datetime(
    fecha_str_, format="%Y %b %d", errors="coerce", cache=True
)

KeyboardInterrupt: 

In [ ]:
station_cols = [
    "CodigoEstacion", "NombreEstacion", "Departamento",
    "Municipio", "ZonaHidrografica", "Latitud", "Longitud"
]

df_daily = (
    precipitacion
    .groupby(station_cols + ["fecha"], as_index=False)
    .agg(
        precip_min_10min=("ValorObservado", "min"),
        precip_max_10min=("ValorObservado", "max"),
        precip_media_10min=("ValorObservado", "mean"),
        precip_acum_diaria=("ValorObservado", "sum")
    )
)

df_daily = df_daily.sort_values(["CodigoEstacion", "fecha"])
print(df_daily.shape)
df_daily.head()

(288051, 12)


,CodigoEstacion,NombreEstacion,Departamento,Municipio,ZonaHidrografica,Latitud,Longitud,fecha,precip_min_10min,precip_max_10min,precip_media_10min,precip_acum_diaria
0,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-19,0.0,0.1,0.000526,0.1
1,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-20,0.0,0.0,0.000000,0.0
2,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-21,0.0,0.1,0.000347,0.1
3,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-22,0.0,0.1,0.000347,0.1
4,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-23,0.0,0.0,0.000000,0.0


In [ ]:
data_2 = pd.read_csv("Precipitación_20251222_2.csv", decimal=",")

for col in ["Latitud", "Longitud", "ValorObservado"]:
    data_2[col] = data_2[col].astype(float)

fecha_str = data_2["FechaObservacion"].str.slice(0, 11)
data_2["fecha"] = pd.to_datetime(
    fecha_str, format="%Y %b %d", errors="coerce", cache=True
)

In [ ]:
df_daily_2 = (
    data_2
    .groupby(station_cols + ["fecha"], as_index=False)
    .agg(
        precip_min_10min=("ValorObservado", "min"),
        precip_max_10min=("ValorObservado", "max"),
        precip_media_10min=("ValorObservado", "mean"),
        precip_acum_diaria=("ValorObservado", "sum")
    )
)

df_daily_2 = df_daily_2.sort_values(["CodigoEstacion", "fecha"])
print(df_daily_2.shape)
df_daily_2.head()

(688910, 12)


,CodigoEstacion,NombreEstacion,Departamento,Municipio,ZonaHidrografica,Latitud,Longitud,fecha,precip_min_10min,precip_max_10min,precip_media_10min,precip_acum_diaria
0,11017020,PR CHOCO: BAGADO,CHOCÓ,BAGADÓ,<nil>,5.412,-76.418,2018-05-04,0.0,3.0,0.147826,3.4
1,11017020,PR CHOCO: BAGADO,CHOCÓ,BAGADÓ,<nil>,5.412,-76.418,2018-05-05,0.0,2.5,0.171429,8.4
2,11017020,PR CHOCO: BAGADO,CHOCÓ,BAGADÓ,<nil>,5.412,-76.418,2018-05-06,0.0,0.0,0.000000,0.0
3,11017020,PR CHOCO: BAGADO,CHOCÓ,BAGADÓ,<nil>,5.412,-76.418,2018-05-07,0.0,6.3,0.206818,9.1
4,11017020,PR CHOCO: BAGADO,CHOCÓ,BAGADÓ,<nil>,5.412,-76.418,2018-05-08,0.0,0.0,0.000000,0.0


## Fuentes adicionales (2021, 2022, 2026)

In [ ]:
def cargar_precipitacion_ideam(nombre_archivo):
    """Carga un CSV IDEAM con columnas en minúscula y lo agrega a nivel diario."""
    df = pd.read_csv(nombre_archivo)
    df = df.rename(columns={
        "codigoestacion":   "CodigoEstacion",
        "nombreestacion":   "NombreEstacion",
        "departamento":     "Departamento",
        "municipio":        "Municipio",
        "zonahidrografica": "ZonaHidrografica",
        "latitud":          "Latitud",
        "longitud":         "Longitud",
        "valorobservado":   "ValorObservado",
        "fechaobservacion": "FechaObservacion",
    })

    for col in ["Latitud", "Longitud", "ValorObservado"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["fecha"] = pd.to_datetime(df["FechaObservacion"], errors="coerce").dt.normalize()

    return (
        df
        .groupby(station_cols + ["fecha"], as_index=False)
        .agg(
            precip_min_10min=("ValorObservado", "min"),
            precip_max_10min=("ValorObservado", "max"),
            precip_media_10min=("ValorObservado", "mean"),
            precip_acum_diaria=("ValorObservado", "sum")
        )
    )

fuentes_adicionales = [
    "precipitacion_2021_1.csv",
    "precipitacion_2021_2.csv",
    "Precipitación_2022.csv",
    "Precipitación_2026_1.csv",
]

df_daily_extra = [cargar_precipitacion_ideam(f) for f in fuentes_adicionales]

for nombre, df_f in zip(fuentes_adicionales, df_daily_extra):
    print(f"{nombre}: {df_f.shape}")

In [ ]:
df_daily_vf = pd.concat([df_daily, df_daily_2] + df_daily_extra)
df_daily_vf = df_daily_vf.drop_duplicates(["CodigoEstacion", "fecha"])
print("df_daily_vf shape:", df_daily_vf.shape)
df_daily_vf.head()

In [ ]:
df_daily_vf['fecha'] = pd.to_datetime(df_daily_vf['fecha'])
df_daily_vf['mes']   = df_daily_vf['fecha'].dt.month

# ── Periodo de referencia fijo (calibración "fuera de muestra") ───────────
# El umbral p95 y la frecuencia histórica de extremos se calibran SOLO con
# datos hasta esta fecha. Así el umbral no se recalcula cada vez que llega
# un mes nuevo (lo que haría que meses ya evaluados cambien de categoría
# retroactivamente sin que haya pasado nada nuevo en ellos). Se usa el
# mismo corte que en "datos panel.ipynb" (panel hídrico municipal) para
# mantener consistencia metodológica entre los dos productos.
FECHA_CORTE_REFERENCIA = pd.Timestamp("2025-12-31")

# ── 1. Percentil 95 (antes 90) por estación y mes — umbral histórico ─────
# Se sube de p90 a p95: un umbral más "ácido"/estricto, que solo marca
# como extremo el 5% de los días más lluviosos históricos de cada
# estación-mes (antes 10%). Se calibra únicamente con el periodo de
# referencia (<= 2025-12-31); ese umbral fijo se aplica luego a TODA la
# serie, incluyendo 2026, de modo que el año corrido se evalúa fuera de
# muestra contra un umbral que no conoce sus propios datos.
df_referencia = df_daily_vf[df_daily_vf['fecha'] <= FECHA_CORTE_REFERENCIA]

p95 = (
    df_referencia
    .groupby(['CodigoEstacion', 'mes'])['precip_acum_diaria']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'precip_acum_diaria': 'p95'})
)

df_daily_vf = df_daily_vf.merge(p95, on=['CodigoEstacion', 'mes'], how='left')
df_daily_vf['extremo'] = (df_daily_vf['precip_acum_diaria'] > df_daily_vf['p95']).astype(int)

# ── 2. Frecuencia histórica de extremos por estación (solo referencia) ───
# Climatología base: se calcula únicamente sobre el periodo de referencia
# para que no quede contaminada por el año en curso (2026), que es
# justamente lo que se quiere evaluar como exceso/déficit más abajo.
station_cols_alerta = ['CodigoEstacion', 'NombreEstacion', 'Departamento', 'Municipio', 'Latitud', 'Longitud']

estaciones_alerta = (
    df_daily_vf[df_daily_vf['fecha'] <= FECHA_CORTE_REFERENCIA]
    .groupby(station_cols_alerta)['extremo']
    .mean()
    .reset_index()
    .rename(columns={'extremo': 'frecuencia_extremos'})
)

# ── 3. Tendencia en los últimos 2 años (regresión mensual) ────────────────
# Ventana amplia (2 años) para que la regresión tenga suficientes puntos
# mensuales y la pendiente sea estadísticamente interpretable; el "año
# corrido" y los "últimos 6 meses" de la sección 5 son la señal específica
# de corto plazo que complementa esta tendencia de más largo plazo.
fecha_max   = df_daily_vf['fecha'].max()
fecha_corte = fecha_max - pd.DateOffset(years=2)
df_reciente = df_daily_vf[df_daily_vf['fecha'] >= fecha_corte].copy()
df_reciente['anio_mes'] = df_reciente['fecha'].dt.to_period('M').dt.to_timestamp()


extremos_mensuales = (
    df_reciente
    .groupby(['CodigoEstacion', 'anio_mes'])['extremo']
    .agg(n_extremos='sum', n_dias='count')
    .reset_index()
)
extremos_mensuales['tasa_extremos'] = extremos_mensuales['n_extremos'] / extremos_mensuales['n_dias']
## Aplicamos una regresion lineal para estimar la pendiente de la tasa de extremos en el tiempo para cada estación. Si la pendiente es positiva y significativa, indicaría una tendencia creciente.
def calcular_tendencia(grupo):
    """Regresión lineal de la tasa mensual de extremos en el tiempo."""
    if len(grupo) < 4:
        return pd.Series({'pendiente': 0.0, 'p_valor': 1.0, 'tendencia': 'insuficiente'})
    x = np.arange(len(grupo))
    y = grupo['tasa_extremos'].values
    slope, _, _, p_value, _ = stats.linregress(x, y)
    if p_value < 0.05:
        direccion = 'creciente' if slope > 0 else 'decreciente'
    else:
        direccion = 'estable'
    return pd.Series({'pendiente': slope, 'p_valor': p_value, 'tendencia': direccion})

tendencias = (
    extremos_mensuales
    .groupby('CodigoEstacion')
    .apply(calcular_tendencia)
    .reset_index()
)

# ── 4. Ratio reciente vs histórico (últimos 2 años) ───────────────────────
freq_reciente = (
    extremos_mensuales
    .groupby('CodigoEstacion')['tasa_extremos']
    .mean()
    .reset_index()
    .rename(columns={'tasa_extremos': 'frecuencia_reciente'})
)

estaciones_alerta = (
    estaciones_alerta
    .merge(tendencias[['CodigoEstacion', 'pendiente', 'p_valor', 'tendencia']], on='CodigoEstacion', how='left')
    .merge(freq_reciente, on='CodigoEstacion', how='left')
)

estaciones_alerta['ratio_reciente'] = (
    estaciones_alerta['frecuencia_reciente']
    / estaciones_alerta['frecuencia_extremos'].replace(0, np.nan)
)

# ── 5. Exceso/déficit del año corrido, con énfasis en los últimos 6 meses ─
# "Estimación de crecimiento" aplicada al año corrido: todo lo posterior al
# corte de referencia (2026) y, en particular, los últimos 6 meses de dato
# disponible. Se compara la tasa de días extremos observada en cada ventana
# contra frecuencia_extremos (climatología histórica <= 2025-12-31) de la
# misma estación. ratio > 1 => exceso de lluvia extrema frente a lo normal;
# ratio < 1 => déficit relativo (menos días extremos que lo histórico). El
# indicador de 6 meses es la señal principal solicitada porque, al ser una
# ventana corta, refleja mejor un cambio reciente que el promedio de 2 años
# de la sección 4.
df_anio_corrido = df_daily_vf[df_daily_vf['fecha'] > FECHA_CORTE_REFERENCIA]

frecuencia_anio_corrido = (
    df_anio_corrido
    .groupby('CodigoEstacion')['extremo']
    .mean()
    .reset_index()
    .rename(columns={'extremo': 'frecuencia_anio_corrido'})
)

fecha_corte_6m  = fecha_max - pd.DateOffset(months=6)
df_ultimos_6m   = df_daily_vf[df_daily_vf['fecha'] >= fecha_corte_6m]

frecuencia_ultimos_6m = (
    df_ultimos_6m
    .groupby('CodigoEstacion')['extremo']
    .mean()
    .reset_index()
    .rename(columns={'extremo': 'frecuencia_ultimos_6m'})
)

estaciones_alerta = (
    estaciones_alerta
    .merge(frecuencia_anio_corrido, on='CodigoEstacion', how='left')
    .merge(frecuencia_ultimos_6m, on='CodigoEstacion', how='left')
)

estaciones_alerta['ratio_anio_corrido'] = (
    estaciones_alerta['frecuencia_anio_corrido']
    / estaciones_alerta['frecuencia_extremos'].replace(0, np.nan)
)
estaciones_alerta['ratio_ultimos_6m'] = (
    estaciones_alerta['frecuencia_ultimos_6m']
    / estaciones_alerta['frecuencia_extremos'].replace(0, np.nan)
)

# +-15% respecto al histórico se lee como exceso/déficit; dentro de esa
# banda se considera variabilidad normal.
UMBRAL_EXCESO_6M  = 1.15
UMBRAL_DEFICIT_6M = 0.85

def clasificar_exceso_deficit(ratio):
    if pd.isna(ratio):
        return 'sin_datos'
    if ratio > UMBRAL_EXCESO_6M:
        return 'EXCESO'
    elif ratio < UMBRAL_DEFICIT_6M:
        return 'DEFICIT'
    else:
        return 'NORMAL'

estaciones_alerta['condicion_lluvia_6m'] = estaciones_alerta['ratio_ultimos_6m'].apply(clasificar_exceso_deficit)

estaciones_alerta.head()


In [ ]:
# ── 6. Alerta compuesta (4 niveles) ───────────────────────────────────────
# Umbral algo más "ácido" que antes: UMBRAL_FREQ se mantiene en 10%, pero
# ahora se mide sobre p95 en vez de p90 (celda anterior), así que la misma
# frecuencia relativa exige días objetivamente más extremos para activar
# la alerta. El criterio de "aceleración" ahora prioriza el ratio de los
# últimos 6 meses (condicion_lluvia_6m == 'EXCESO') sobre el ratio de los
# últimos 2 años, porque es la señal que mejor captura si el exceso es un
# fenómeno reciente y no un promedio diluido en 2 años; el ratio de 2 años
# se conserva como señal de respaldo cuando la de 6 meses no es concluyente.
UMBRAL_FREQ   = 0.10   # >10 % de días históricos sobre p95 (climatología <= 2025-12-31)
UMBRAL_RATIO  = 1.10   # últimos 2 años con ≥10 % más extremos que el histórico (señal de respaldo)

def categorizar_alerta(row):
    alta_freq      = row['frecuencia_extremos'] > UMBRAL_FREQ
    creciente      = row['tendencia'] == 'creciente'
    aceleracion_6m = row['condicion_lluvia_6m'] == 'EXCESO'
    aceleracion_2a = pd.notna(row['ratio_reciente']) and row['ratio_reciente'] > UMBRAL_RATIO
    aceleracion    = aceleracion_6m or aceleracion_2a

    if alta_freq and (creciente or aceleracion):
        return 'CRÍTICA'
    elif alta_freq:
        return 'ALTA'
    elif creciente or aceleracion:
        return 'MODERADA'
    else:
        return 'BAJA'

# Solo categoría textual
estaciones_alerta['alerta lluvias'] = estaciones_alerta.apply(categorizar_alerta, axis=1)

# ── 7. Resumen ────────────────────────────────────────────────────────────
print("=== Distribución de alerta compuesta ===")
print(estaciones_alerta['alerta lluvias'].value_counts())
print("\n=== Tendencias últimos 2 años ===")
print(estaciones_alerta['tendencia'].value_counts())
print("\n=== Condición de lluvia últimos 6 meses (año corrido, vs. histórico <= 2025-12-31) ===")
print(estaciones_alerta['condicion_lluvia_6m'].value_counts())

cols_resumen = [
    'CodigoEstacion', 'NombreEstacion', 'Departamento', 'Municipio',
    'frecuencia_extremos', 'frecuencia_reciente', 'ratio_reciente',
    'tendencia', 'p_valor',
    'ratio_anio_corrido', 'ratio_ultimos_6m', 'condicion_lluvia_6m',
    'alerta lluvias'
]
estaciones_alerta[cols_resumen].sort_values('alerta lluvias', ascending=False).head(5)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# INDICADOR DE SEQUÍA – Días Consecutivos Secos (CDD)
# ══════════════════════════════════════════════════════════════════════════════

UMBRAL_DIA_SECO = 1.0  # < 1 mm/día = día seco (estándar OMM/IDEAM)

# ── 1. Extraer rachas de días consecutivos secos por estación ─────────────
def extraer_rachas_secas(grupo):
    """Devuelve DataFrame (CodigoEstacion, fecha_inicio, duracion_dias) por racha seca."""
    cod   = grupo['CodigoEstacion'].iloc[0]
    grupo = grupo.sort_values('fecha').reset_index(drop=True)
    rachas, racha, inicio = [], 0, None
    for _, row in grupo.iterrows():
        if row['precip_acum_diaria'] < UMBRAL_DIA_SECO:
            if racha == 0:
                inicio = row['fecha']
            racha += 1
        else:
            if racha > 0:
                rachas.append({'fecha_inicio': inicio, 'duracion_dias': racha})
            racha = 0
    if racha > 0:
        rachas.append({'fecha_inicio': inicio, 'duracion_dias': racha})
    if rachas:
        df = pd.DataFrame(rachas)
    else:
        df = pd.DataFrame({'fecha_inicio': pd.Series(dtype='datetime64[ns]'),
                           'duracion_dias': pd.Series(dtype='int64')})
    df['CodigoEstacion'] = cod
    return df

rachas_por_estacion = (
    df_daily_vf
    .groupby('CodigoEstacion', group_keys=False)
    .apply(extraer_rachas_secas)
    .reset_index(drop=True)
)
rachas_por_estacion['duracion_dias'] = pd.to_numeric(rachas_por_estacion['duracion_dias'])

# ── 2. Percentil 90 histórico de duración de rachas por estación ──────────
p90_rachas = (
    rachas_por_estacion
    .groupby('CodigoEstacion')['duracion_dias']
    .quantile(0.90)
    .reset_index()
    .rename(columns={'duracion_dias': 'p90_racha'})
)

rachas_por_estacion = rachas_por_estacion.merge(p90_rachas, on='CodigoEstacion', how='left')
rachas_por_estacion['sequia_extrema'] = (
    rachas_por_estacion['duracion_dias'] > rachas_por_estacion['p90_racha']
).astype(int)

# ── 3. Frecuencia histórica de rachas extremas por estación ───────────────
freq_sequia_hist = (
    rachas_por_estacion
    .groupby('CodigoEstacion')['sequia_extrema']
    .mean()
    .reset_index()
    .rename(columns={'sequia_extrema': 'freq_sequia_hist'})
)

# ── 4. Frecuencia reciente (últimos 2 años) de rachas extremas ────────────
fecha_max_s   = df_daily_vf['fecha'].max()
fecha_corte_s = fecha_max_s - pd.DateOffset(years=2)

rachas_rec = rachas_por_estacion[
    rachas_por_estacion['fecha_inicio'] >= fecha_corte_s
].copy()

freq_sequia_rec = (
    rachas_rec
    .groupby('CodigoEstacion')['sequia_extrema']
    .mean()
    .reset_index()
    .rename(columns={'sequia_extrema': 'freq_sequia_rec'})
)

# ── 5. Tendencia mensual (regresión lineal) ───────────────────────────────
rachas_rec['anio_mes'] = rachas_rec['fecha_inicio'].dt.to_period('M').dt.to_timestamp()

def tasa_mensual_sequia(df_racha):
    cod = df_racha['CodigoEstacion'].iloc[0]
    result = (
        df_racha
        .groupby('anio_mes')['sequia_extrema']
        .mean()
        .reset_index()
        .rename(columns={'sequia_extrema': 'tasa_sequia'})
    )
    result['CodigoEstacion'] = cod
    return result

tasa_mensual_por_est = (
    rachas_rec
    .groupby('CodigoEstacion', group_keys=False)
    .apply(tasa_mensual_sequia)
    .reset_index(drop=True)
)

def calcular_tendencia_sequia(grupo):
    if len(grupo) < 4:
        return pd.Series({'pendiente_seq': 0.0, 'tendencia_sequia': 'insuficiente'})
    x = np.arange(len(grupo))
    y = grupo['tasa_sequia'].values
    slope, _, _, p_value, _ = stats.linregress(x, y)
    direccion = 'estable'
    if p_value < 0.05:
        direccion = 'creciente' if slope > 0 else 'decreciente'
    return pd.Series({'pendiente_seq': slope, 'tendencia_sequia': direccion})

tendencias_sequia = (
    tasa_mensual_por_est
    .groupby('CodigoEstacion')
    .apply(calcular_tendencia_sequia)
    .reset_index()
)

# ── 6. Ratio reciente vs histórico de sequías ─────────────────────────────
ratio_sequia = freq_sequia_hist.merge(freq_sequia_rec, on='CodigoEstacion', how='left')
ratio_sequia['ratio_sequia'] = (
    ratio_sequia['freq_sequia_rec']
    / ratio_sequia['freq_sequia_hist'].replace(0, np.nan)
)

# ── 7. Categorizar sequía ─────────────────────────────────────────────────
UMBRAL_FREQ_SEQ  = 0.10
UMBRAL_RATIO_SEQ = 1.20

estaciones_sequia = (
    freq_sequia_hist
    .merge(ratio_sequia[['CodigoEstacion', 'freq_sequia_rec', 'ratio_sequia']], on='CodigoEstacion', how='left')
    .merge(tendencias_sequia[['CodigoEstacion', 'tendencia_sequia']], on='CodigoEstacion', how='left')
)

def categorizar_sequia(row):
    alta_freq   = row['freq_sequia_hist'] > UMBRAL_FREQ_SEQ
    creciente   = row['tendencia_sequia'] == 'creciente'
    aceleracion = pd.notna(row['ratio_sequia']) and row['ratio_sequia'] > UMBRAL_RATIO_SEQ
    if alta_freq and (creciente or aceleracion):
        return 'SEVERA'
    elif alta_freq:
        return 'MODERADA'
    elif creciente or aceleracion:
        return 'LEVE'
    else:
        return 'NORMAL'


estaciones_sequia['sequia_categoria'] = estaciones_sequia.apply(categorizar_sequia, axis=1)

# ── 8. Incorporar sequía en estaciones_alerta ─────────────────────────────
estaciones_alerta = estaciones_alerta.merge(
    estaciones_sequia[['CodigoEstacion', 'sequia_categoria']],
    on='CodigoEstacion', how='left'
)

estaciones_alerta['sequia_categoria'] = estaciones_alerta['sequia_categoria'].fillna('NORMAL')
#estaciones_alerta['sequia_nivel']     = estaciones_alerta['sequia_nivel'].fillna(0).astype(int)

# ── 9. Resumen ────────────────────────────────────────────────────────────
#print("=== Alerta lluvia (estaciones) ===")
#print(estaciones_alerta['alerta_lluvias'].value_counts())
#print("\n=== Categoría sequía (estaciones) ===")
#print(estaciones_alerta['sequia_categoria'].value_counts())

In [ ]:
# Carga davipola y proyecta a EPSG 3116
davipola = pd.read_excel(os.path.join(MENSUALES_DIR, "davipola_dane.xlsx"))
def modo_alerta(serie):
    return modo_seguro(serie, default='BAJA')

# Misma condición que en "datos panel.ipynb": a cada municipio se le
# extrapola la información de la estación más cercana, con un radio
# máximo de 50 km. Municipios sin ninguna estación dentro de ese radio
# quedan sin fila en el panel municipal (no se inventa dato).
RADIO_MAX_KM = 50.0

gdf_mun = gpd.GeoDataFrame(
    davipola,
    geometry=gpd.points_from_xy(davipola.LONGITUD, davipola.LATITUD),
    crs="EPSG:4326",
).to_crs(epsg=3116)

# Convierte estaciones_alerta a GeoDataFrame y proyecta
gdf_estaciones = gpd.GeoDataFrame(
    estaciones_alerta,
    geometry=gpd.points_from_xy(estaciones_alerta.Longitud, estaciones_alerta.Latitud),
    crs="EPSG:4326"
).to_crs(epsg=3116)

# Asigna a cada MUNICIPIO la estación más cercana (no al revés), limitada
# a RADIO_MAX_KM. how='left' conserva todos los municipios; los que no
# tienen ninguna estación dentro del radio quedan con columnas de estación
# en NaN y se descartan después (igual que en el panel hídrico mensual).
gdf_est_muni = gpd.sjoin_nearest(
    gdf_mun[['COD_DPTO', 'NOM_DPTO', 'COD_MPIO', 'NOM_MPIO', 'geometry']],
    gdf_estaciones,
    how='left',
    max_distance=RADIO_MAX_KM * 1000,
    distance_col="dist_m"
)

n_mun_total = gdf_mun['COD_MPIO'].nunique()
n_mun_sin_estacion = gdf_est_muni.loc[gdf_est_muni['CodigoEstacion'].isna(), 'COD_MPIO'].nunique()
gdf_est_muni = gdf_est_muni.dropna(subset=['CodigoEstacion'])
print(f"Municipios con estación asignada (radio <= {RADIO_MAX_KM:.0f} km): "
      f"{n_mun_total - n_mun_sin_estacion} / {n_mun_total}")

# ── Helpers robustos ante NaN ──────────────────────────────────────────────
def modo_seguro(serie, default='BAJA'):
    vc = serie.dropna().value_counts()
    return vc.idxmax() if not vc.empty else default

def tendencia_muni(serie):
    vals = serie.dropna()
    if vals.empty:
        return 'sin_datos'
    if 'creciente' in vals.values:
        return 'creciente'
    vc = vals.value_counts()
    return vc.idxmax() if not vc.empty else 'sin_datos'

def modo_sequia(serie):
    return modo_seguro(serie, default='NORMAL')

def modo_condicion_lluvia(serie):
    # Si alguna estación del municipio está en EXCESO o DEFICIT, esa señal
    # prevalece sobre NORMAL (igual criterio que tendencia_muni: la
    # condición más extrema del municipio manda, no el promedio).
    vals = serie.dropna()
    if vals.empty:
        return 'sin_datos'
    if 'EXCESO' in vals.values:
        return 'EXCESO'
    if 'DEFICIT' in vals.values:
        return 'DEFICIT'
    vc = vals.value_counts()
    return vc.idxmax() if not vc.empty else 'NORMAL'

# ── Agrega alertas a nivel municipal ──────────────────────────────────────
df_alerta_municipal = (
    gdf_est_muni
    .groupby(['COD_DPTO', 'NOM_DPTO', 'COD_MPIO', 'NOM_MPIO'], as_index=False)
    .agg(
        frecuencia_extremos=('frecuencia_extremos', 'max'),
        frecuencia_reciente=('frecuencia_reciente', 'max'),
        ratio_reciente=('ratio_reciente', 'max'),
        tendencia=('tendencia', tendencia_muni),
        alerta_lluvias=('alerta lluvias', modo_alerta),   # 👈 AQUÍ
        ratio_anio_corrido=('ratio_anio_corrido', 'max'),
        ratio_ultimos_6m=('ratio_ultimos_6m', 'max'),
        condicion_lluvia_6m=('condicion_lluvia_6m', modo_condicion_lluvia),
        sequia_categoria=('sequia_categoria', modo_sequia),
        n_estaciones=('CodigoEstacion', 'count'),
        distancia_km_estacion_asignada=('dist_m', lambda s: s.min() / 1000),
    )
)
estaciones_alerta = estaciones_alerta.rename(columns={'alerta lluvias': 'alerta_lluvias'})

# Derivar columnas numéricas desde las categóricas (consistencia garantizada)
NIVEL_NUMERICO = {'BAJA': 0, 'MODERADA': 1, 'ALTA': 2, 'CRÍTICA': 3}
NIVEL_SEQUIA   = {'NORMAL': 0, 'LEVE': 1, 'MODERADA': 2, 'SEVERA': 3}

df_alerta_municipal['sequia_nivel'] = df_alerta_municipal['sequia_categoria'].map(NIVEL_SEQUIA)

# ── Cruce con municipios históricamente afectados por inundaciones ─────────
flood_path = os.path.join(BASE_DIR, "Datos Procesados", "municipios_afectados_ola_invernal.xlsx")
flood = pd.read_excel(flood_path)[['cod_divipola', 'municipio_afectado']]
df_alerta_municipal = df_alerta_municipal.merge(
    flood, left_on='COD_MPIO', right_on='cod_divipola', how='left'
).drop(columns='cod_divipola')
df_alerta_municipal['municipio_afectado'] = df_alerta_municipal['municipio_afectado'].fillna(0).astype(int)

n_flood = (df_alerta_municipal['municipio_afectado'] == 1).sum()
print(f"Municipios con antecedente de inundación: {n_flood:,}")

print(f"Municipios con cobertura de estaciones: {len(df_alerta_municipal):,}")
print("\n=== Alerta lluvia municipal (alerta_compuesta) ===")
print("\n=== Categoría sequía municipal (sequia_categoria) ===")
print(df_alerta_municipal['sequia_categoria'].value_counts())
print("\n=== Condición de lluvia municipal, últimos 6 meses (exceso/déficit) ===")
print(df_alerta_municipal['condicion_lluvia_6m'].value_counts())

# ── Guardar salidas ───────────────────────────────────────────────────────
# 1. Nivel estación → consumido por datos precipitacion diarios.ipynb
estaciones_alerta['cod_norm'] = estaciones_alerta['CodigoEstacion'].astype(str).str.strip().str.lstrip('0')
out_estaciones = os.path.join(WEB_DATA_DIR, "alerta_historica_estaciones.csv")
estaciones_alerta.to_csv(out_estaciones, index=False, encoding="utf-8-sig")
print("\nGuardado (estaciones):", out_estaciones)

# 2. Nivel municipal → salida principal de este notebook
out_municipal = os.path.join(WEB_DATA_DIR, "alerta_historica_municipal.csv")
df_alerta_municipal.to_csv(out_municipal, index=False, encoding="utf-8-sig")
print("Guardado (municipal) :", out_municipal)
df_alerta_municipal.head()


In [ ]:
# ── Genera GeoJSON de alerta histórica municipal para el mapa web ─────────
# Reemplaza la fuente del mapa "Lluvias" del index: en vez del layer diario
# (municipios_alertas.geojson, generado por datos precipitacion diarios.ipynb
# a partir de df_muni), el mapa pasa a leer directamente este dataframe
# (df_alerta_municipal), que ya aplica el radio máximo de 50 km por
# municipio calculado arriba.
import requests, zipfile, io

geo_cache = os.path.join(WEB_DATA_DIR, "municipios_colombia_geo.gpkg")

if os.path.exists(geo_cache):
    print("Cargando geometría desde caché local...")
    gdf_geo_hist = gpd.read_file(geo_cache)
else:
    print("Descargando polígonos GADM Colombia nivel 2 (primera vez)...")
    url = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_COL_2.json.zip"
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        fname = next(f for f in z.namelist() if f.endswith(".json"))
        with z.open(fname) as f:
            gdf_geo_hist = gpd.read_file(f)
    gdf_geo_hist = gdf_geo_hist.to_crs("EPSG:4326")
    gdf_geo_hist["geometry"] = gdf_geo_hist["geometry"].simplify(tolerance=0.005, preserve_topology=True)
    gdf_geo_hist = gdf_geo_hist[["NAME_1", "NAME_2", "geometry"]].copy()
    gdf_geo_hist.to_file(geo_cache, driver="GPKG")
    print(f"  → {len(gdf_geo_hist):,} polígonos guardados en caché ({geo_cache})")

# ── Asignar COD_MPIO (código DANE) a polígonos GADM si aún no está ────────
if "COD_MPIO" not in gdf_geo_hist.columns:
    print("Asignando códigos DANE a polígonos GADM via sjoin...")
    gdf_mun_4326 = gdf_mun.to_crs("EPSG:4326")[["COD_MPIO", "geometry"]].copy()

    joined = gpd.sjoin(gdf_mun_4326, gdf_geo_hist[["geometry"]], how="left", predicate="within")
    code_map = (
        joined.dropna(subset=["index_right"])
        .groupby("index_right")["COD_MPIO"]
        .first()
    )
    gdf_geo_hist["COD_MPIO"] = gdf_geo_hist.index.map(code_map)

    missing = gdf_geo_hist["COD_MPIO"].isna()
    if missing.any():
        near = gpd.sjoin_nearest(
            gdf_geo_hist[missing][["geometry"]].reset_index(),
            gdf_mun_4326.reset_index(drop=True),
            how="left",
        ).drop_duplicates("index").set_index("index")["COD_MPIO"]
        gdf_geo_hist.loc[missing, "COD_MPIO"] = gdf_geo_hist[missing].index.map(near)

    gdf_geo_hist["COD_MPIO"] = pd.to_numeric(gdf_geo_hist["COD_MPIO"], errors="coerce").apply(
        lambda x: str(int(x)) if pd.notna(x) else ""
    ).str.strip()
    gdf_geo_hist.to_file(geo_cache, driver="GPKG")
    print(f"  → COD_MPIO asignado ({gdf_geo_hist['COD_MPIO'].notna().sum():,} polígonos) y caché actualizado")

gdf_geo_hist["COD_MPIO"] = pd.to_numeric(gdf_geo_hist["COD_MPIO"], errors="coerce").apply(
    lambda x: str(int(x)) if pd.notna(x) else ""
).str.strip()

# ── Une la geometría con df_alerta_municipal (izquierda = TODOS los polígonos,
# así los municipios sin estación dentro de 50 km siguen apareciendo en el mapa
# pero marcados como SIN_DATOS) ────────────────────────────────────────────
df_alerta_hist_geo = df_alerta_municipal.copy()
df_alerta_hist_geo["COD_MPIO"] = df_alerta_hist_geo["COD_MPIO"].astype(str).str.strip()

cols_join_hist = [
    "COD_MPIO", "NOM_MPIO", "NOM_DPTO",
    "alerta_lluvias", "condicion_lluvia_6m", "sequia_categoria",
    "frecuencia_extremos", "frecuencia_reciente", "ratio_ultimos_6m",
    "distancia_km_estacion_asignada", "n_estaciones", "municipio_afectado",
]
df_alerta_hist_geo = df_alerta_hist_geo[[c for c in cols_join_hist if c in df_alerta_hist_geo.columns]]

gdf_hist_out = gdf_geo_hist.merge(df_alerta_hist_geo, on="COD_MPIO", how="left")
matched_hist = gdf_hist_out["alerta_lluvias"].notna().sum()
print(f"Polígonos con alerta histórica asignada (radio <= {RADIO_MAX_KM:.0f} km): "
      f"{matched_hist:,} / {len(gdf_hist_out):,}")

# ── Rellenar faltantes (municipios sin estación dentro del radio) ─────────
gdf_hist_out["alerta_lluvias"]      = gdf_hist_out["alerta_lluvias"].fillna("SIN_DATOS")
gdf_hist_out["condicion_lluvia_6m"] = gdf_hist_out["condicion_lluvia_6m"].fillna("sin_datos")
gdf_hist_out["sequia_categoria"]    = gdf_hist_out["sequia_categoria"].fillna("NORMAL")
for col in ["frecuencia_extremos", "frecuencia_reciente", "ratio_ultimos_6m",
            "distancia_km_estacion_asignada", "n_estaciones"]:
    gdf_hist_out[col] = pd.to_numeric(gdf_hist_out[col], errors="coerce").fillna(0)
gdf_hist_out["municipio_afectado"] = pd.to_numeric(gdf_hist_out["municipio_afectado"], errors="coerce").fillna(0).astype(int)
gdf_hist_out["NOM_MPIO"] = gdf_hist_out["NOM_MPIO"].fillna(gdf_hist_out["NAME_2"])
gdf_hist_out["NOM_DPTO"] = gdf_hist_out["NOM_DPTO"].fillna(gdf_hist_out["NAME_1"])

# ── Exportar GeoJSON ───────────────────────────────────────────────────────
keep_hist = [
    "NAME_1", "NAME_2", "COD_MPIO", "NOM_MPIO", "NOM_DPTO",
    "alerta_lluvias", "condicion_lluvia_6m", "sequia_categoria",
    "frecuencia_extremos", "frecuencia_reciente", "ratio_ultimos_6m",
    "distancia_km_estacion_asignada", "n_estaciones", "municipio_afectado",
    "geometry",
]
gdf_hist_final = gdf_hist_out[[c for c in keep_hist if c in gdf_hist_out.columns]]

out_geo_hist = os.path.join(WEB_DATA_DIR, "alerta_historica_municipal.geojson")
gdf_hist_final.to_file(out_geo_hist, driver="GeoJSON")
print(f"\nGeoJSON exportado: {out_geo_hist}")
print(f"  Tamaño: {os.path.getsize(out_geo_hist) / 1024:.0f} KB")
print("\n=== Distribución alerta_lluvias en el mapa ===")
print(gdf_hist_final["alerta_lluvias"].value_counts())
gdf_hist_final.drop(columns="geometry").head()
